In [1]:
import sys

assert sys.version_info >= (3, 10)

In [2]:
import torch
from packaging.version import Version

assert Version(torch.__version__) >=Version("2.6.0")

In [4]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [5]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "Physics_Informed_Digital_Twin_for_Hydraulic_Systems"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, fig_extension="png", tight_layout=True, resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [7]:
import deepxde as dde
from deepxde import utils
import numpy as np

dde.config.set_random_seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [8]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")

Set the default float type to float64
Backend: pytorch


Defining Geometry and Time domain- Transient Window( a restricted interval of time within a larger temporal domain used to train the network)

In [9]:
pipe_length = 10.0 #0 to 10 meters
time_window = 1.0 # 0 to 1 sec

geom = dde.geometry.Interval(0, pipe_length)
timedomain = dde.geometry.TimeDomain(0, time_window)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

Defining the Navier-Stokes Governing Physics Equation

In [11]:
def fluid_momentum_pde(x, y):
    P = y[:, 0:1]
    v = y[:, 1:2]

    dP_dx = dde.grad.jacobian(y, x, i=0, j=0)
    dv_dt = dde.grad.jacobian(y, x, i=1, j=1)
    dv_dx = dde.grad.jacobian(y, x, i=1, j=0)

    #Fluid material properties(Hydraulic Oil constants)
    rho = 850.0 # Density (kg/m^3)
    nu = 0.00004 # kinematic viscosity

    d2v_dx2 = dde.grad.hessian(y, x, component=1, i=0,j=0)

    # 1D Navier-Stokes momentum residual equation
    momentum_residual = dv_dt + v * dv_dx + (1.0 / rho) * dP_dx - nu * d2v_dx2
    return momentum_residual

Define the Boundary and Initial conditions

In [16]:
def inlet_boundary(x, on_boundary):
    return on_boundary and np.isclose(x, 0.0)

def outlet_boundary(x, on_boundary):
    return on_boundary and np.isclose(x, pipe_length)

inlet_velocity_bc = dde.icbc.DirichletBC(geomtime, lambda x: 2.5, inlet_boundary, component=0)
outlet_pressure_bc = dde.icbc.DirichletBC(geomtime, lambda x: 10.0,outlet_boundary, component=0)
# as we have mentioned earlier first column to pressure and 2nd to velocity
# we will call component=0 for pressure and component=1 for velocity

initial_velocity = dde.IC(geomtime, lambda x: 0.0,lambda _, on_initial: on_initial, component=1)
initial_pressure = dde.IC(geomtime, lambda x: 100.0,lambda _, on_initial: on_initial, component=0)